In [1]:
import pdfplumber

In [2]:
all_text=" "
with pdfplumber.open("sample.pdf") as pdf:
    for page in pdf.pages:
        all_text+=page.extract_text() +"\n"
   

In [5]:
print(all_text)

 UNIT 1: Computational Models
The Concept of Computational Model :- The computer architecture and language classes must have a
common foundation or paradigm called a Computational Model. The concept of a computational model represents
a higher level of abstraction than either the computer architecture or the programming language alone, and covers
both, as show below -
Computational Model
Level of
Abstraction
Computer Computer
Architecture Language
Interpretation of the computational model concept as a high-level abstraction.
The Concept of Computation Model - The concept of computational model comprises the set of the following
three abstractions –
1. The basic items of computations
2. The problem description model
3. The execution model
Contrary to initial thoughts, the set of abstractions that should be chosen to specify computational models is far from
obvious. A smaller number of criteria would define fewer but more basic computational models, while a larger
number of criteria woul

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\admin\Desktop\project\rag-implementation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=60,add_start_index=True)

In [5]:
chunks=text_splitter.split_text(all_text)

In [9]:
print(f"number of chunks:{len(chunks)}")

number of chunks:80


In [10]:
print(chunks[79])

performed on a per-thread basis.
Threads have a similar life cycle to the processes and are mainly managed in the same way. Initially each process is
created with a single thread. However, threads are usually allowed to create new ones using particular system calls.
Then, a thread tree is typically created for each process.
Process
Thread tree.


In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import SparseEncoder
import torch

from qdrant_client import QdrantClient

from qdrant_client.http import models as rest

from dotenv import load_dotenv

import os

load_dotenv()

dense_model=SentenceTransformer("all-MiniLM-L6-v2")


client = QdrantClient(
    url=os.getenv("Qdrant_url"),
    prefer_grpc=False 
    
)

client.recreate_collection(
    collection_name=os.getenv("collection_name"),
    vectors_config={
      "dense":rest.VectorParams(size=384, distance=rest.Distance.COSINE)
      }
)

for index,chunk in enumerate  (chunks):
  dense_embedding=dense_model.encode(chunk)
  client.upsert(
     collection_name=os.getenv("collection_name"),
     points=[
        rest.PointStruct(
         id=index,
         vector={
           "dense":dense_embedding.tolist(),
                 },
         payload={"text":chunk}

       )
     ] 
   )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 298.37it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 204/204 [00:00<00:00, 387.47it/s, Materializing param=cls.predictions.transform.dense.weight]                 
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.prediction

Query portion

In [ ]:
Query="What is a computational model?"
dense_query_embedding=(dense_model.encode(Query)).tolist()


In [13]:
print(type(client))


<class 'qdrant_client.qdrant_client.QdrantClient'>


In [14]:
import qdrant_client
print(qdrant_client.__file__)


c:\Users\admin\Desktop\project\rag-implementation\.venv\lib\site-packages\qdrant_client\__init__.py


In [ ]:
result=client.query_points(
    collection_name=os.getenv("collection_name"),
    query=dense_query_embedding,
    limit=5,
    with_payload=True
)

In [16]:
print(result)

points=[ScoredPoint(id=0, version=1, score=0.7930154, payload={'text': 'UNIT 1: Computational Models\nThe Concept of Computational Model :- The computer architecture and language classes must have a\ncommon foundation or paradigm called a Computational Model. The concept of a computational model represents\na higher level of abstraction than either the computer architecture or the programming language alone, and covers\nboth, as show below -\nComputational Model\nLevel of\nAbstraction\nComputer Computer\nArchitecture Language'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=2, score=0.73031145, payload={'text': 'Abstraction\nComputer Computer\nArchitecture Language\nInterpretation of the computational model concept as a high-level abstraction.\nThe Concept of Computation Model - The concept of computational model comprises the set of the following\nthree abstractions –\n1. The basic items of computations\n2. The problem description model\n3. The execution mo

In [17]:
points=result.points

In [18]:
texts=[]
for point in points:
    payload=point.payload
    text=payload["text"]
    texts.append(text)
print(texts)

['UNIT 1: Computational Models\nThe Concept of Computational Model :- The computer architecture and language classes must have a\ncommon foundation or paradigm called a Computational Model. The concept of a computational model represents\na higher level of abstraction than either the computer architecture or the programming language alone, and covers\nboth, as show below -\nComputational Model\nLevel of\nAbstraction\nComputer Computer\nArchitecture Language', 'Abstraction\nComputer Computer\nArchitecture Language\nInterpretation of the computational model concept as a high-level abstraction.\nThe Concept of Computation Model - The concept of computational model comprises the set of the following\nthree abstractions –\n1. The basic items of computations\n2. The problem description model\n3. The execution model\nContrary to initial thoughts, the set of abstractions that should be chosen to specify computational models is far from', 'programming languages and are implemented by memory or 

In [19]:
context="\n\n".join(texts)
print(context)

UNIT 1: Computational Models
The Concept of Computational Model :- The computer architecture and language classes must have a
common foundation or paradigm called a Computational Model. The concept of a computational model represents
a higher level of abstraction than either the computer architecture or the programming language alone, and covers
both, as show below -
Computational Model
Level of
Abstraction
Computer Computer
Architecture Language

Abstraction
Computer Computer
Architecture Language
Interpretation of the computational model concept as a high-level abstraction.
The Concept of Computation Model - The concept of computational model comprises the set of the following
three abstractions –
1. The basic items of computations
2. The problem description model
3. The execution model
Contrary to initial thoughts, the set of abstractions that should be chosen to specify computational models is far from

programming languages and are implemented by memory or register addresses in ar

In [20]:
prompt_template=""""
you are an expert assistant trained to explain answer given context.

Context:{context}

Question:{question}

Answer:
"""

In [21]:
prompt=prompt_template.format(
    context=context,
    question=Query
)

In [ ]:
from google import genai
client=genai.Client(api_key=os.getenv("api_key"))
response=client.models.generate_content(model="gemini-3-flash-preview",
                               contents=prompt)

print(response.text)








Based on the provided text, a **computational model** is defined as a high-level abstraction that serves as a common foundation or paradigm for both computer architecture and programming languages. It sits at a higher level of abstraction than either of these components individually and encompasses both.

Specifically, a computational model consists of a set of three key abstractions:

1.  **Basic items of computations:** These are the fundamental units the model works with, such as data (as seen in Turing or von Neumann models), objects/messages, or functions.
2.  **Problem description model:** This defines how a problem or its solution is described, typically using either a **procedural style** (stating how to solve the problem) or a **declarative style** (stating what the problem is).
3.  **Execution model:** This outlines how the computation is performed, including the interpretation of computation, the semantics of the process, and the control of the execution sequence.

In practi